In [13]:
import math
import os
import time

import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

# -----------------------------
# USER INPUTS
# -----------------------------
shapefile_path = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\Exports\BasePolys.shp"
ghi_raster = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\Rasters\Non-Arctic_Average_GHI_y2005_2024_5070.tif"
log_file = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\GHI\GHI_Log.txt"
output_csv = r"C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\GHI\CVG_GHI.csv"
chunk_size = 10000

# -----------------------------
# Logging
# -----------------------------
def log(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Remove old CSV if exists
# -----------------------------
if os.path.exists(output_csv):
    os.remove(output_csv)
    log("Existing output CSV removed.")

# -----------------------------
# Load shapefile
# -----------------------------
log(f"Loading shapefile: {shapefile_path}")
gdf = gpd.read_file(shapefile_path)
gdf = gdf.set_crs(epsg=26916)
# Ensure required fields exist
required_fields = ["ROW_ID", "Square_Met"]
for field in required_fields:
    if field not in gdf.columns:
        raise ValueError(f"Missing required field: {field}")

total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# Compute chunks
# -----------------------------
# num_chunks = math.ceil(total_polygons / chunk_size)
# num_chunks = 6
# log(f"Processing in {num_chunks} chunks of {chunk_size} polygons each.")

# -----------------------------
# Process chunks
# -----------------------------
# for chunk_idx in range(num_chunks):
#     start_idx = chunk_idx * chunk_size
#     end_idx = min((chunk_idx + 1) * chunk_size, total_polygons)
#     chunk_gdf = gdf.iloc[0:0].copy()

#     log(f"Processing chunk {chunk_idx + 1}/{num_chunks} ({start_idx} to {end_idx - 1})...")
#     start_time = time.time()

    # Zonal statistics
stats = zonal_stats(
    gdf,
    ghi_raster,
    stats=["mean"],
    all_touched=True,
    nodata=-9999
)

# Add GHI_mean
gdf["GHI_mean"] = [s["mean"] for s in stats]

# Keep only desired columns
output_df = gdf[["ROW_ID", "Square_Met", "GHI_mean"]]

# Append to CSV
# if chunk_idx == 0:
output_df.to_csv(output_csv, index=False, mode="w")
# else:
#     output_df.to_csv(output_csv, index=False, mode="a", header=False)

log("All chunks processed successfully.")
log(f"Final CSV saved to: {output_csv}")

Existing output CSV removed.
Loading shapefile: C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\Exports\BasePolys.shp
Loaded 6 polygons.
All chunks processed successfully.
Final CSV saved to: C:\Users\KevinWilliams\Documents\Source Files\CVG ROW Solar\GHI\CVG_GHI.csv
